# Real-Time Sign Language Recognition: Initial MLP Prototype

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelBinarizer
import json

print("TensorFlow version:", tf.__version__)

## 1. Data Generation / Simulation

In [ ]:
CLASSES = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']
NUM_CLASSES = len(CLASSES)
SAMPLES_PER_CLASS = 1000

def create_hand_template(sign_name):
    # Base coordinates for 21 hand landmarks (approximate hand shape centered at wrist [0, 0, 0])
    # Landmarks index:
    # 0: Wrist, 1-4: Thumb, 5-8: Index, 9-12: Middle, 13-16: Ring, 17-20: Pinky
    landmarks = np.zeros((21, 3))
    
    # Finger base (MCP) joints default positions
    landmarks[0] = [0.0, 0.0, 0.0]        # Wrist
    landmarks[1] = [-0.15, -0.05, 0.0]     # Thumb MCP
    landmarks[5] = [-0.08, -0.22, 0.0]     # Index MCP
    landmarks[9] = [-0.02, -0.24, 0.0]     # Middle MCP
    landmarks[13] = [0.03, -0.23, 0.0]     # Ring MCP
    landmarks[17] = [0.08, -0.21, 0.0]     # Pinky MCP

    # Helper to extend a finger upwards (y decreases in standard screen coordinates)
    def extend_finger(mcp_idx, landmarks):
        offset_x = (mcp_idx - 9) * 0.005 # slight fan out
        landmarks[mcp_idx + 1] = [landmarks[mcp_idx][0] + offset_x, landmarks[mcp_idx][1] - 0.07, -0.01]
        landmarks[mcp_idx + 2] = [landmarks[mcp_idx + 1][0] + offset_x, landmarks[mcp_idx + 1][1] - 0.06, -0.02]
        landmarks[mcp_idx + 3] = [landmarks[mcp_idx + 2][0] + offset_x, landmarks[mcp_idx + 2][1] - 0.05, -0.03]
        
    # Helper to fold a finger down into a fist
    def fold_finger(mcp_idx, landmarks):
        landmarks[mcp_idx + 1] = [landmarks[mcp_idx][0], landmarks[mcp_idx][1] - 0.03, 0.04]
        landmarks[mcp_idx + 2] = [landmarks[mcp_idx][0], landmarks[mcp_idx][1] - 0.01, 0.06]
        landmarks[mcp_idx + 3] = [landmarks[mcp_idx][0], landmarks[mcp_idx][1] + 0.01, 0.05]
        
    if sign_name == 'A': # Fist with thumb on side
        for idx in [5, 9, 13, 17]: fold_finger(idx, landmarks)
        landmarks[2] = [-0.22, -0.12, 0.02]  # Thumb extended outward
        landmarks[3] = [-0.25, -0.16, 0.01]
        landmarks[4] = [-0.24, -0.20, -0.01]
    elif sign_name == 'B': # Flat open hand
        for idx in [5, 9, 13, 17]: extend_finger(idx, landmarks)
        # Thumb tucked in front of palm
        landmarks[2] = [-0.08, -0.09, 0.02]
        landmarks[3] = [-0.04, -0.12, 0.03]
        landmarks[4] = [-0.02, -0.14, 0.03]
    elif sign_name == 'C': # Curved open claw
        for idx in [5, 9, 13, 17]:
            # Curved fingers
            landmarks[idx + 1] = [landmarks[idx][0], landmarks[idx][1] - 0.05, 0.05]
            landmarks[idx + 2] = [landmarks[idx][0] - 0.02, landmarks[idx][1] - 0.10, 0.08]
            landmarks[idx + 3] = [landmarks[idx][0] - 0.05, landmarks[idx][1] - 0.12, 0.06]
        # Curved thumb
        landmarks[2] = [-0.18, -0.08, 0.03]
        landmarks[3] = [-0.16, -0.14, 0.05]
        landmarks[4] = [-0.10, -0.18, 0.04]
    elif sign_name == 'D': # Index extended, others form loop with thumb
        extend_finger(5, landmarks) # Index up
        for idx in [9, 13, 17]: fold_finger(idx, landmarks) # others fold
        # Thumb touches middle finger tip
        landmarks[2] = [-0.10, -0.08, 0.02]
        landmarks[3] = [-0.05, -0.11, 0.02]
        landmarks[4] = [-0.02, -0.12, 0.01]
    elif sign_name == 'L': # L shape: Index up, thumb out
        extend_finger(5, landmarks)
        for idx in [9, 13, 17]: fold_finger(idx, landmarks)
        # Thumb extended fully out
        landmarks[2] = [-0.22, -0.08, 0.0]
        landmarks[3] = [-0.28, -0.10, -0.01]
        landmarks[4] = [-0.32, -0.10, -0.02]
    elif sign_name == 'Y': # Pinky and Thumb out
        extend_finger(17, landmarks) # Pinky extended
        for idx in [5, 9, 13]: fold_finger(idx, landmarks)
        # Thumb extended out
        landmarks[2] = [-0.22, -0.08, 0.0]
        landmarks[3] = [-0.28, -0.10, -0.01]
        landmarks[4] = [-0.32, -0.10, -0.02]
    elif sign_name == 'I': # Pinky only
        extend_finger(17, landmarks)
        for idx in [5, 9, 13]: fold_finger(idx, landmarks)
        # Fold thumb
        landmarks[2] = [-0.10, -0.08, 0.02]
        landmarks[3] = [-0.08, -0.10, 0.03]
        landmarks[4] = [-0.06, -0.11, 0.03]
    elif sign_name == 'V': # Peace/V sign
        extend_finger(5, landmarks)
        extend_finger(9, landmarks)
        landmarks[8][0] -= 0.03 # spread index left
        landmarks[12][0] += 0.03 # spread middle right
        for idx in [13, 17]: fold_finger(idx, landmarks)
        # Fold thumb
        landmarks[2] = [-0.10, -0.08, 0.02]
        landmarks[3] = [-0.08, -0.10, 0.03]
        landmarks[4] = [-0.06, -0.11, 0.03]
    elif sign_name == 'W': # Index, Middle, Ring extended, Pinky/Thumb folded
        for idx in [5, 9, 13]: extend_finger(idx, landmarks)
        fold_finger(17, landmarks)
        # Fold thumb
        landmarks[2] = [-0.10, -0.08, 0.02]
        landmarks[3] = [-0.08, -0.10, 0.03]
        landmarks[4] = [-0.06, -0.11, 0.03]
    elif sign_name == 'F': # OK sign: Thumb & Index tip touch, Middle, Ring, Pinky out
        for idx in [9, 13, 17]: extend_finger(idx, landmarks)
        # Thumb and Index tip touch
        landmarks[2] = [-0.12, -0.08, 0.02]
        landmarks[3] = [-0.09, -0.13, 0.03]
        landmarks[4] = [-0.06, -0.16, 0.03] # Thumb tip
        landmarks[6] = [-0.08, -0.14, 0.02]
        landmarks[7] = [-0.07, -0.16, 0.03]
        landmarks[8] = [-0.06, -0.16, 0.03] # Index tip touches thumb tip
    else:
        # Fallback for remaining letters to create a unique static synthetic signature
        # We uniquely fold/extend based on the ASCII value of the character
        seed_val = ord(sign_name)
        for idx in [5, 9, 13, 17]:
            if (seed_val % idx) % 2 == 0:
                extend_finger(idx, landmarks)
            else:
                fold_finger(idx, landmarks)
    return landmarks

# Generate dataset with noise and rotation
X = []
y = []

np.random.seed(42)
for label_idx, name in enumerate(CLASSES):
    template = create_hand_template(name)
    for _ in range(SAMPLES_PER_CLASS):
        sample = template.copy()
        
        # 1. Add small noise to each point (jitter)
        noise = np.random.normal(0, 0.006, sample.shape)
        sample += noise
        
        # 2. Add random uniform scaling
        scale = np.random.uniform(0.85, 1.15)
        sample *= scale
        
        # 3. Add random rotation
        theta = np.random.uniform(-0.15, 0.15) # in radians
        c, s = np.cos(theta), np.sin(theta)
        R = np.array([
            [c, -s, 0],
            [s,  c, 0],
            [0,  0, 1]
        ])
        sample = np.dot(sample, R.T)
        
        # 4. Flatten coordinates to 63-dim vector
        X.append(sample.flatten())
        y.append(label_idx)

X = np.array(X)
y = np.array(y)
print(f"Dataset generated: {X.shape[0]} samples, {X.shape[1]} features (21 landmarks x 3 axes).")

## 2. Preprocessing & Data Splits

In [ ]:
lb = LabelBinarizer()
y_encoded = lb.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y)
print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

## 3. Neural Network Design (Keras Sequential)

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(63,)),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.15),
    keras.layers.Dense(64, activation='relu'),
    keras.layers.Dropout(0.1),
    keras.layers.Dense(32, activation='relu'),
    keras.layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 4. Train Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=25,
    batch_size=32,
    validation_data=(X_test, y_test),
    verbose=1
)

## 5. Evaluate and Save to Disk

In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {test_acc*100:.2f}%")

# Save model locally
model.save('sign_language_model.h5')

# Save labels configuration
with open('labels.json', 'w') as f:
    json.dump(CLASSES, f)